In [14]:
import numpy as np
import pandas as pd

Cambio de estrategia, la idea ahora es crear una sola base que tenga lo que necesitamos resumido para poder luego hacer un append, buscamos una base con el ingreso individual mensual ajustado por cargas familiares 

In [15]:
# Carga base de datos con ipc
data_externa = pd.read_excel("data_externa.xlsx", sheet_name='datos')

# 1990

In [16]:
data = pd.read_stata(r"Z:\survey\ECU\ENEMDU\1990\m11\data_orig\ECU_1990m11.dta")

In [17]:
data['salbasic'] = data['salbasic'].replace(999999, np.nan)
data['salbasic'] = data['salbasic'].replace(999998, np.nan)
data['ingr'] = data['salbasic']

In [18]:
data['ingr_ene'] = data.apply(lambda x: x['salbasic'] if x['ene'] == 'ocupado' else None, axis=1)
data['ingr_feb'] = data.apply(lambda x: x['salbasic'] if x['feb'] == 'ocupado' else None, axis=1)
data['ingr_mar'] = data.apply(lambda x: x['salbasic'] if x['mar'] == 'ocupado' else None, axis=1)
data['ingr_abr'] = data.apply(lambda x: x['salbasic'] if x['abr'] == 'ocupado' else None, axis=1)
data['ingr_may'] = data.apply(lambda x: x['salbasic'] if x['may'] == 'ocupado' else None, axis=1)
data['ingr_jun'] = data.apply(lambda x: x['salbasic'] if x['jun'] == 'ocupado' else None, axis=1)
data['ingr_jul'] = data.apply(lambda x: x['salbasic'] if x['jul'] == 'ocupado' else None, axis=1)
data['ingr_ago'] = data.apply(lambda x: x['salbasic'] if x['ago'] == 'ocupado' else None, axis=1)
data['ingr_sep'] = data.apply(lambda x: x['salbasic'] if x['sep'] == 'ocupado' else None, axis=1)
data['ingr_oct'] = data.apply(lambda x: x['salbasic'] if x['oct'] == 'ocupado' else None, axis=1)
data['ingr_nov'] = data.apply(lambda x: x['salbasic'] if x['nov'] == 'ocupado' else None, axis=1)
data['ingr_dic'] = data.apply(lambda x: x['salbasic'] if x['dic'] == 'ocupado' else None, axis=1)

Data externa

In [19]:
# filtra año de interés
datos_actual = data_externa[data_externa['Año'] == 1990]
datos_base = data_externa[data_externa['Año'] == 2014]

In [20]:
ipc_dict = {fila['trimestre']: {
        'Nacional': fila['Nacional'],
        'Guayaquil': fila['Guayaquil'],
        'Quito': fila['Quito'],
        'Cuenca': fila['Cuenca']
    }
     for _, fila in datos_actual.iterrows()
     }

ipc_base_dict = {fila['trimestre']: {
        'Nacional': fila['Nacional'],
        'Guayaquil': fila['Guayaquil'],
        'Quito': fila['Quito'],
        'Cuenca': fila['Cuenca']
    }
     for _, fila in datos_base.iterrows()
     }

tipo_cambio_dict = dict(zip(datos_actual['trimestre'], datos_actual['tipo de cambio']))

In [21]:
# Corregimos los códigos para usarlos cómo texto
data['ciudad'] = data['ciudad'].apply(str)
data['ciudad'] = data['ciudad'].apply(lambda x: '0' + x if len(x) == 5 else x)

data['ciudad_2'] = data['ciudad'].apply(lambda x: x[:2])

Diccionario ciudades disponibles

In [22]:
parroquia_dict = {
    '01': 'Cuenca',
    '09': 'Guayaquil',
    '17': 'Quito'
}

data['ciudad_asignada'] = data['ciudad_2'].apply(lambda x: parroquia_dict.get(x, 'Nacional'))

In [23]:
# Función que asigna valores correspondientes
def asigna_ipc(fila, trimestre):
    return ipc_dict.get(trimestre, {}).get(fila['ciudad_asignada'], None)

def asigna_ipc_base(fila, trimestre):
    return ipc_base_dict.get(trimestre, {}).get(fila['ciudad_asignada'], None)

In [24]:
data['ipc_t1'] = data.apply(lambda fila: asigna_ipc(fila, 1), axis=1)
data['ipc_base_t1'] = data.apply(lambda fila: asigna_ipc_base(fila, 1), axis=1)
data['tipo_cambio_t1'] = tipo_cambio_dict.get(1)

data['ipc_t2'] = data.apply(lambda fila: asigna_ipc(fila, 2), axis=1)
data['ipc_base_t2'] = data.apply(lambda fila: asigna_ipc_base(fila, 2), axis=1)
data['tipo_cambio_t2'] = tipo_cambio_dict.get(2)

data['ipc_t3'] = data.apply(lambda fila: asigna_ipc(fila, 3), axis=1)
data['ipc_base_t3'] = data.apply(lambda fila: asigna_ipc_base(fila, 3), axis=1)
data['tipo_cambio_t3'] = tipo_cambio_dict.get(3)

data['ipc_t4'] = data.apply(lambda fila: asigna_ipc(fila, 4), axis=1)
data['ipc_base_t4'] = data.apply(lambda fila: asigna_ipc_base(fila, 4), axis=1)
data['tipo_cambio_t4'] = tipo_cambio_dict.get(4)

In [25]:
# Calculamos el deflactor
data['def_t1'] = (data['ipc_base_t1'] / data['ipc_t1'])
data['def_t2'] = (data['ipc_base_t2'] / data['ipc_t2'])
data['def_t3'] = (data['ipc_base_t3'] / data['ipc_t3'])
data['def_t4'] = (data['ipc_base_t4'] / data['ipc_t4'])

In [26]:
# Ingreso real por mes
data['ingr_ene_r'] = (data['ingr_ene'] / data['tipo_cambio_t1']) * data['def_t1']
data['ingr_feb_r'] = (data['ingr_feb'] / data['tipo_cambio_t1']) * data['def_t1']
data['ingr_mar_r'] = (data['ingr_mar'] / data['tipo_cambio_t1']) * data['def_t1']
data['ingr_abr_r'] = (data['ingr_abr'] / data['tipo_cambio_t2']) * data['def_t2']
data['ingr_may_r'] = (data['ingr_may'] / data['tipo_cambio_t2']) * data['def_t2']
data['ingr_jun_r'] = (data['ingr_jun'] / data['tipo_cambio_t2']) * data['def_t2']
data['ingr_jul_r'] = (data['ingr_jul'] / data['tipo_cambio_t3']) * data['def_t3']
data['ingr_ago_r'] = (data['ingr_ago'] / data['tipo_cambio_t3']) * data['def_t3']
data['ingr_sep_r'] = (data['ingr_sep'] / data['tipo_cambio_t3']) * data['def_t3']
data['ingr_oct_r'] = (data['ingr_oct'] / data['tipo_cambio_t4']) * data['def_t4']
data['ingr_nov_r'] = (data['ingr_nov'] / data['tipo_cambio_t4']) * data['def_t4']
data['ingr_dic_r'] = (data['ingr_dic'] / data['tipo_cambio_t4']) * data['def_t4']

In [70]:
w = data['fexp']

# Weighted average for ingr_ene_r
x = data['ingr_ene_r']
mask = x.notna() & w.notna()
weighted_avg_ene = (x[mask] * w[mask]).sum() / w[mask].sum()
# or equivalently: weighted_avg_ene = np.average(x[mask], weights=w[mask])

# Weighted average for ingr_feb_r
x = data['ingr_feb_r']
mask = x.notna() & w.notna()
weighted_avg_feb = (x[mask] * w[mask]).sum() / w[mask].sum()
# or equivalently: weighted_avg_feb = np.average(x[mask], weights=w[mask])

# Weighted average for ingr_mar_r
x = data['ingr_mar_r']
mask = x.notna() & w.notna()
weighted_avg_mar = (x[mask] * w[mask]).sum() / w[mask].sum()
# or equivalently: weighted_avg_mar = np.average(x[mask], weights=w[mask])

# Weighted average for ingr_abr_r
x = data['ingr_abr_r']
mask = x.notna() & w.notna()
weighted_avg_abr = (x[mask] * w[mask]).sum() / w[mask].sum()
# or equivalently: weighted_avg_abr = np.average(x[mask], weights=w[mask])

# Weighted average for ingr_may_r
x = data['ingr_may_r']
mask = x.notna() & w.notna()
weighted_avg_may = (x[mask] * w[mask]).sum() / w[mask].sum()
# or equivalently: weighted_avg_may = np.average(x[mask], weights=w[mask])

# Weighted average for ingr_jun_r
x = data['ingr_jun_r']
mask = x.notna() & w.notna()
weighted_avg_jun = (x[mask] * w[mask]).sum() / w[mask].sum()
# or equivalently: weighted_avg_jun = np.average(x[mask], weights=w[mask])

# Weighted average for ingr_jul_r
x = data['ingr_jul_r']
mask = x.notna() & w.notna()
weighted_avg_jul = (x[mask] * w[mask]).sum() / w[mask].sum()
# or equivalently: weighted_avg_jul = np.average(x[mask], weights=w[mask])

# Weighted average for ingr_ago_r
x = data['ingr_ago_r']
mask = x.notna() & w.notna()
weighted_avg_ago = (x[mask] * w[mask]).sum() / w[mask].sum()
# or equivalently: weighted_avg_ago = np.average(x[mask], weights=w[mask])

# Weighted average for ingr_sep_r
x = data['ingr_sep_r']
mask = x.notna() & w.notna()
weighted_avg_sep = (x[mask] * w[mask]).sum() / w[mask].sum()
# or equivalently: weighted_avg_sep = np.average(x[mask], weights=w[mask])

# Weighted average for ingr_oct_r
x = data['ingr_oct_r']
mask = x.notna() & w.notna()
weighted_avg_oct = (x[mask] * w[mask]).sum() / w[mask].sum()
# or equivalently: weighted_avg_oct = np.average(x[mask], weights=w[mask])

# Weighted average for ingr_nov_r
x = data['ingr_nov_r']
mask = x.notna() & w.notna()
weighted_avg_nov = (x[mask] * w[mask]).sum() / w[mask].sum()
# or equivalently: weighted_avg_nov = np.average(x[mask], weights=w[mask])

# Weighted average for ingr_dic_r
x = data['ingr_dic_r']
mask = x.notna() & w.notna()
weighted_avg_dic = (x[mask] * w[mask]).sum() / w[mask].sum()
# or equivalently: weighted_avg_dic = np.average(x[mask], weights=w[mask])

Calculo ingreso de los hogares

In [27]:
columnas_idef = ['rn', 'estrato', 'ciudad', 'zona', 'sector', 'conglo', 'area', 'vivienda', 'hogar']

data['idef_hogar'] = data[columnas_idef].astype(str).agg(''.join, axis=1)
len(data['idef_hogar'].unique())

7910

Si todos los miembros del hogar tienen NA como ingreso, mantener NA, si al menos uno tiene un ingreso sumamos para el ingreso del hogar, así evitamos subestimar el ingreso del hogar si tenemos valores perdidos

In [28]:
# Definimos una función que sume pero devuelva NA si todos son NA
def sum_with_na(series):
    if series.isna().all():
        return pd.NA
    else:
        return series.sum(skipna=True)

In [31]:
data['ingr_ene_h'] = data.groupby('idef_hogar')['ingr_ene_r'].transform(sum_with_na)
data['ingr_feb_h'] = data.groupby('idef_hogar')['ingr_feb_r'].transform(sum_with_na)
data['ingr_mar_h'] = data.groupby('idef_hogar')['ingr_mar_r'].transform(sum_with_na)
data['ingr_abr_h'] = data.groupby('idef_hogar')['ingr_abr_r'].transform(sum_with_na)
data['ingr_may_h'] = data.groupby('idef_hogar')['ingr_may_r'].transform(sum_with_na)
data['ingr_jun_h'] = data.groupby('idef_hogar')['ingr_jun_r'].transform(sum_with_na)
data['ingr_jul_h'] = data.groupby('idef_hogar')['ingr_jul_r'].transform(sum_with_na)
data['ingr_ago_h'] = data.groupby('idef_hogar')['ingr_ago_r'].transform(sum_with_na)
data['ingr_sep_h'] = data.groupby('idef_hogar')['ingr_sep_r'].transform(sum_with_na)
data['ingr_oct_h'] = data.groupby('idef_hogar')['ingr_oct_r'].transform(sum_with_na)
data['ingr_nov_h'] = data.groupby('idef_hogar')['ingr_nov_r'].transform(sum_with_na)
data['ingr_dic_h'] = data.groupby('idef_hogar')['ingr_dic_r'].transform(sum_with_na)

Ingreso individual descontando cargas familiares

Utilizando la metodología del autor dividimos el ingreso del hogar para la escala $(A_{i}+kC_{i})^{s}$ donde $A_{i}$ es al número de adultos, $C_{i}$ es el número de niños en el hogar $i$. $k$ es el costo en recursos de cada niño y $s$ busca reflejar las restricciones

In [33]:
k = 0.4
s = 0.9

In [34]:
# Si es necesario calcular el número de niños
data['es_nino'] = data['edad'] < 10

data['ninos'] = data.groupby('idef_hogar')['es_nino'].transform('sum')

# Si es necesario calcular el número de adultos
data['es_adulto'] = data['edad'] > 10

data['adultos'] = data.groupby('idef_hogar')['es_adulto'].transform('sum')

C:\Users\oscarj\AppData\Local\Temp\ipykernel_34248\418205408.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data['es_adulto'] = data['edad'] > 10
C:\Users\oscarj\AppData\Local\Temp\ipykernel_34248\418205408.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data['adultos'] = data.groupby('idef_hogar')['es_adulto'].transform('sum')


In [35]:
data['escala'] = (data['adultos'] + k * data['ninos']) ** s

C:\Users\oscarj\AppData\Local\Temp\ipykernel_34248\1776195318.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data['escala'] = (data['adultos'] + k * data['ninos']) ** s


In [75]:
for x in ['ingr_ene_h', 'ingr_feb_h', 'ingr_mar_h', 'ingr_abr_h', 'ingr_may_h', 'ingr_jun_h', 'ingr_jul_h', 'ingr_ago_h', 'ingr_sep_h', 'ingr_oct_h', 'ingr_nov_h', 'ingr_dic_h' ]:
    data[f'{x}_i'] = data[f'{x}'] / data['escala']

C:\Users\oscarj\AppData\Local\Temp\ipykernel_34248\1314487705.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data[f'{x}_i'] = data[f'{x}'] / data['escala']
C:\Users\oscarj\AppData\Local\Temp\ipykernel_34248\1314487705.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data[f'{x}_i'] = data[f'{x}'] / data['escala']
C:\Users\oscarj\AppData\Local\Temp\ipykernel_34248\1314487705.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance

In [77]:
data['ingr_abr_h_i'].mean()

509.34409313452295

In [78]:
# Weighted average for ingr_dic_r
x = data['ingr_abr_h_i']
mask = x.notna() & w.notna()
weighted_avg_abr = (x[mask] * w[mask]).sum() / w[mask].sum()
# or equivalently: weighted_avg_dic = np.average(x[mask], weights=w[mask])

In [79]:
weighted_avg_abr

522.7067843468766